In [70]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import sys

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error,  mean_squared_error,r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn

from torch.utils.data import DataLoader ,TensorDataset


In [26]:
powerplant_data = pd.read_csv('powerplant_data.csv')
print(powerplant_data)
powerplant_data.shape
powerplant_data.columns
powerplant_data.dtypes
# AT => temperature
# V => vacuum
# AP => pressure
# RH => humidity
# PE => produced energy

         AT      V       AP     RH      PE
0      8.34  40.77  1010.84  90.01  480.48
1     23.64  58.49  1011.40  74.20  445.75
2     29.74  56.90  1007.15  41.91  438.76
3     19.07  49.69  1007.22  76.79  453.09
4     11.80  40.66  1017.13  97.20  464.43
...     ...    ...      ...    ...     ...
9563  15.12  48.92  1011.80  72.93  462.59
9564  33.41  77.95  1010.30  59.72  432.90
9565  15.99  43.34  1014.20  78.66  465.96
9566  17.65  59.87  1018.58  94.65  450.93
9567  23.68  51.30  1011.86  71.24  451.67

[9568 rows x 5 columns]


AT    float64
V     float64
AP    float64
RH    float64
PE    float64
dtype: object

In [61]:
powerplant_data.isnull().sum()

X = powerplant_data.drop('PE', axis=1, inplace=False)
y = powerplant_data['PE']

X_train ,  X_test, y_train, y_test = train_test_split(X, y , test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

np.set_printoptions(threshold=100)
y_train.value_counts()
X_train_scaled


array([[ 0.74805289,  0.72006931, -0.32660017, -0.49711722],
       [ 0.86181948,  1.26515721, -0.98521113,  0.8181501 ],
       [ 0.93409473,  1.52314975,  0.32523844,  0.80167494],
       ...,
       [-0.22097078, -0.834965  ,  0.36756563, -0.83554456],
       [ 0.94747903,  1.14245344, -0.41971997, -0.45455637],
       [-1.77355014, -1.19049131,  1.92520594,  0.91837402]],
      shape=(7654, 4))

In [67]:
X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float64)
X_test_tensor =  torch.tensor(X_test_scaled, dtype = torch.float64)

y_train_tensor = torch.tensor(y_train.values, dtype = torch.float64 ).view(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float64 ).reshape(-1,1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32,shuffle=True )
test_loader = DataLoader(test_dataset, batch_size=32)


In [69]:

class Artifficial_Neural_network(nn.Module):
    def __init__(self):
        super(Artifficial_Neural_network , self).__init__()

        self.model = nn.Sequential(
            # 1 hidden  layers
            nn.Linear(in_features = X_test_tensor.shape[1],out_features=6),
            nn.ReLU(),

            # 2nd Hidden_layers 
            nn.Linear(in_features=6, out_features=6),
            nn.ReLU(), # activations functions apply on hidden layers
            nn.Linear(in_features=6, out_features=1) #out layers
            
            

            
        )
    
    